In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Vanishing Gradient Problem — Section 5

Explore how gradient magnitudes shrink through deep networks for **Sigmoid, Tanh, Softplus, ReLU, and Hinge** activations.

The gradient at layer $l$ from the output is the product of all intermediate derivatives:
$$\frac{\partial L}{\partial W^{(1)}} \propto \prod_{k=2}^{L} f'(x_k)$$
If $|f'(x)| < 1$ at every layer, this product shrinks exponentially — the **vanishing gradient problem**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def sigmoid(x):     return 1 / (1 + np.exp(-np.clip(x, -30, 30)))
def d_sigmoid(x):   s = sigmoid(x); return s * (1 - s)
def d_tanh(x):      return 1 - np.tanh(x) ** 2
def softplus(x):    return np.log1p(np.exp(np.clip(x, -30, 30)))
def d_softplus(x):  return sigmoid(x)
def d_relu(x):      return (x > 0).astype(float)
def hinge(x):       return np.maximum(0, 1 - x)
def d_hinge(x):     return np.where(x < 1, -1.0, 0.0)

ACTS = {
    'Sigmoid':  (sigmoid,           d_sigmoid,  '#2563eb'),
    'Tanh':     (np.tanh,           d_tanh,     '#059669'),
    'Softplus': (softplus,          d_softplus, '#d97706'),
    'ReLU':     (lambda x: np.maximum(0,x), d_relu, '#dc2626'),
    'Hinge':    (hinge,             d_hinge,    '#7c3aed'),
}


def draw_vanishing(x0=0.5, depth=20):
    xs = np.linspace(-4, 4, 300)
    depths = np.arange(1, depth + 1)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    for name, (fn, dfn, c) in ACTS.items():
        axes[0].plot(xs, fn(xs), color=c, lw=2, label=name)
        axes[1].plot(xs, np.abs(dfn(xs)), color=c, lw=2, label=name)
        gv = abs(float(dfn(np.array([x0]))[0]))
        axes[2].semilogy(depths, np.maximum(1e-20, gv ** depths), color=c, lw=2, label=f'{name} ({gv:.3f})')

    for ax in axes[:2]:
        ax.axvline(x0, color='k', lw=1.5, ls='--', alpha=0.6)
    axes[2].axhline(1e-7, color='red', lw=1, ls='--', alpha=0.5, label='≈ 0')

    axes[0].set_title('Activation f(x)', fontsize=11)
    axes[0].set_xlabel('x'); axes[0].legend(fontsize=9)
    axes[1].set_title("|f′(x)| — gradient magnitude per layer", fontsize=11)
    axes[1].set_xlabel('x'); axes[1].set_ylim(-0.05, 1.1); axes[1].legend(fontsize=9)
    axes[2].set_title(f'Gradient magnitude after N layers  (at x={x0:.2f})', fontsize=11)
    axes[2].set_xlabel('Depth N'); axes[2].set_ylabel('|gradient| (log scale)')
    axes[2].legend(fontsize=9, title="|f′(x0)|")
    axes[2].grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\nAt x = {x0:.2f}:")
    for name, (fn, dfn, c) in ACTS.items():
        gv = abs(float(dfn(np.array([x0]))[0]))
        g20 = gv ** 20
        status = '✓ stable' if gv >= 0.9 else '✗ vanishes' if gv < 0.1 else '⚠ shrinks'
        print(f"  {name:<10} |f′|={gv:.4f}  after 20 layers: {g20:.2e}  {status}")


widgets.interact(
    draw_vanishing,
    x0=widgets.FloatSlider(min=-4, max=4, step=0.1, value=0.5, description='Input x', continuous_update=False),
    depth=widgets.IntSlider(min=2, max=50, step=1, value=20, description='Max depth', continuous_update=False),
)